In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

The function `logging` is used as a *decorator*.  It takes a function 
`f` as its argument and returns a new function `logged_f` that returns 
the same result as the function `f`, but additionally it prints its arguments 
before the function is called and when the function returns, both the function 
call and the result is printed.

The *decorator* `logging` is useful for debugging.

In [ ]:
function logging<F extends (...args: any[]) => any>(f: F): F {
  const logged_f = ((...args: Parameters<F>): ReturnType<F> => {
    const r = f(...args);
    return r;
  }) as F;
  return logged_f;
}

# An Array-Based Implementation of Quick-Sort

The function $\texttt{swap}(L, x, y)$ swaps the elements at index $x$ and $y$ in $L$.

In [ ]:
function swap(L: number[], i: number, j: number): void {
  const temp = L[i];
  L[i] = L[j];
  L[j] = temp;
}

The function $\texttt{partition}(\texttt{start}, \texttt{end}, L)$ returns an index $m$ into the array $L$ and 
regroups the elements of $L$ such that after the function returns the following holds:
 
  - $\forall i \in \{\texttt{start}, \cdots, m-1\} : L[i] \leq L[m]$,
  - $\forall i \in \{ m+1, \cdots, \texttt{end} \}  : L[m] <    L[i]$,
  - $L[m] = \texttt{pivot}$.
  
Here, `pivot` is the element that is at the index `end` at the time of the invocation 
of the function, i.e. we have

  - $L[\texttt{end}] = \texttt{pivot}$
  
at invocation time.
  
The for-loop of `partition` maintains the following invariants:

 - $\forall i \in \{\texttt{start}, \cdots, \texttt{left} \} : L[i] \leq \texttt{pivot}$,
 - $\forall i \in \{\texttt{left}+1, \cdots, \texttt{idx}-1\} : \texttt{pivot} < L[i]$,
 - $L[\texttt{end}] = \texttt{pivot}$.

These invariants are depicted below:

![Invariants for partitioning](lomuto.png)

This algorithm has been suggested by *Nico Lomuto*.  It is not the most efficient implementation of `partition`, but
it is easier to understand than the algorithm given by *Tony Hoare* that uses two separate loops.

In [ ]:
const partition = logging(function partition(start: number, end: number, L: number[]): number {
  const pivot = L[end];
  let left = start - 1;
  for (let idx = start; idx < end; idx++) {
    if (L[idx] <= pivot) {
      left += 1;
      swap(L, left, idx);
    }
  }
  swap(L, left + 1, end);
  return left + 1;
});

The function `quickSort(start, end, L)` sorts the subarray `L[start:end+1]` in place.

In [ ]:
function quickSort(start: number, end: number, L: number[]): void {
  if (end <= start) return; // nothing to do, single element
  const m = partition(start, end, L);
  quickSort(start, m - 1, L);
  quickSort(m + 1, end, L);
}

The function $\texttt{sort}(L)$ sorts the array $L$ in place.

In [ ]:
function sort(L: number[]): void {
  quickSort(0, L.length - 1, L);
}

## Testing

In [ ]:
function demo() {
  const L: number[] = Array.from({ length: 19 }, () => Math.floor(Math.random() * 99) + 1);
  console.log("L =", L);

  let S = [...L];
  sort(S);
  console.log("S =", S);
}

In [ ]:
demo();

In [ ]:
function isOrdered(L: number[]): void {
  for (let i = 0; i < L.length - 1; i++) {
    if (L[i] > L[i + 1]) {
      throw new Error(`${L} not ordered at ${i}`);
    }
  }
}

In [ ]:
function sameElements(L: number[], S: number[]): void {
  const count = (arr: number[]) => {
    const map = new Map<number, number>();
    for (const x of arr) map.set(x, (map.get(x) ?? 0) + 1);
    return map;
  };

  const cL = count(L);
  const cS = count(S);

  console.assert(cL.size === cS.size, "Different number of unique elements");
  for (const [k, v] of cL.entries()) {
    console.assert(cS.get(k) === v, `Mismatch in count for element ${k}`);
  }
}

The function $\texttt{testSort}(n, k)$ generates $n$ random arrays of length $k$, sorts them, and checks whether the output is sorted and contains the same elements as the input.

In [ ]:
function testSort(n: number, k: number): void {
  for (let i = 0; i < n; i++) {
    const L = Array.from({ length: k }, () => randomIntRange(0, 2 * k));
    const oldL = [...L];
    sort(L);
    isOrdered(L);
    sameElements(oldL, L);
    console.assert(L.length === oldL.length, "Array length changed");
    process.stdout.write(".");
  }
  console.log("\nAll tests successful!");
}

In [ ]:
console.time("testSort");
testSort(100, 20000);
console.timeEnd("testSort");

Next, we sort a million random integers.

In [ ]:
console.time("1 million random integers");
const k = 1_000_000;
const L1 = Array.from({ length: k }, () => randomIntRange(0, 1000));
sort(L1);
console.timeEnd("1 million random integers");

Next, we sort a hundred thousand integers.  This time, many of the integers have the same value.

In [ ]:
const L2 = Array.from({ length: 100_000 }, () => randomIntRange(0, 100));
console.time("100k integers (with repetitions)");
sort(L2);
console.timeEnd("100k integers (with repetitions)");

Finally, we test the worst case and sort 5000 integers that are already sorted in ascending order.  
Since quicksort is recursive, this represents the worst possible recursion pattern.  
In TypeScript, there is `no adjustable recursion limit` as in *Python*,  
but if the recursion depth becomes too large, the program will eventually throw  
a `RangeError: Maximum call stack size exceeded`.  
However, for arrays of this size, this is typically not an issue.

In [ ]:
const L3 = Array.from({ length: 5000 }, (_, i) => i);
console.time("worst case (sorted array)");
sort(L3);
console.timeEnd("worst case (sorted array)");

If we *shuffle* the array that is to be sorted before calling `sort`, the worst case behaviour disappears.

In [ ]:
for (let i = L3.length - 1; i > 0; i--) {
  const j = Math.floor(Math.random() * (i + 1));
  [L3[i], L3[j]] = [L3[j], L3[i]];
}

In [ ]:
console.time("shuffled array");
sort(L3);
console.timeEnd("shuffled array");